In [ ]:
import os
import pandas as pd
import shutil

In [ ]:
# load corpus

df = pd.read_csv(
    "polityka_SECONDROUND_final_articles.csv",
    low_memory=False
)
df.head()

In [ ]:
# publication years
n_articles = len(df)

df["publish_date"] = pd.to_datetime(
    df["publish_date"],
    errors="coerce",
    utc=True
)

year_counts = (

    df["publish_date"]
    .dt.year
    .value_counts()
    .sort_index()

)

# categories

category_counts = (

    df["category"]
    .value_counts()

)

n_categories = df["category"].nunique()

# text length

text_length = (

    df["text"]
    .fillna("")
    .str.len()

)

text_length_summary = text_length.describe()

# image count

image_count = (

    df["image_url"]
    .fillna("")
    .apply(

        lambda x:

        len(

            [

                url

                for url in str(x).split("|")

                if url.strip()

            ]

        )

    )

)

image_count_summary = image_count.describe()

# unique images

unique_images = set()

for urls in df["image_filename"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if url:

            unique_images.add(url)

n_unique_images = len(unique_images)

# print summary
print("articles")
print(n_articles)

print("categories")
display(category_counts)

print("number of categories")
print(n_categories)

print("text length")
print(text_length_summary)

print("image count")
print(image_count_summary)

print("unique images")
print(n_unique_images)

In [ ]:
# load corpus

df1 = pd.read_csv(
    "rp_SECONDROUND_final_articles.csv",
    low_memory=False
)
df1.head()

In [ ]:
df1 = df1.rename(columns={'image_urls': 'image_url',
    'image_filenames': 'image_filename'})

In [ ]:
# publication years
n_articles = len(df1)

category_counts = (

    df1["category"]
    .value_counts()

)

n_categories = df1["category"].nunique()

# text length

text_length = (

    df1["text"]
    .fillna("")
    .str.len()

)

text_length_summary = text_length.describe()

# image count

image_count = (

    df1["image_url"]
    .fillna("")
    .apply(

        lambda x:

        len(

            [

                url

                for url in str(x).split("|")

                if url.strip()

            ]

        )

    )

)

image_count_summary = image_count.describe()

# unique images

unique_images = set()

for urls in df1["image_filename"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if url:

            unique_images.add(url)

n_unique_images = len(unique_images)

# print summary
print("articles")
print(n_articles)

print("categories")
display(category_counts)

print("number of categories")
print(n_categories)

print("text length")
print(text_length_summary)

print("image count")
print(image_count_summary)

print("unique images")
print(n_unique_images)

In [ ]:
# List of paths to the 8 datasets
file_paths = [
    "elmundo_SECONDROUND_final_articles.csv",
    "eldiario_SECONDROUND_final_articles.csv",
    "rp_SECONDROUND_final_articles.csv",
    "polityka_SECONDROUND_final_articles.csv",
    "magyar_SECONDROUND_final_articles.csv",
    "telex_SECONDROUND_final_articles.csv",
    "spiegel_SECONDROUND_final_articles.csv",
    "welt_SECONDROUND_final_articles.csv",
]

datasets = {}

for path in file_paths:
    df = pd.read_csv(path)
    datasets[path] = df

    print(path)
    print(list(df.columns))

In [ ]:
for path in file_paths:
    df = pd.read_csv(path)
    datasets[path] = df
    print(path)
    print(df.shape)


In [ ]:
import pandas as pd
import hashlib
import ast

df = pd.read_csv("telex_SECONDROUND_final_articles.csv")

def get_urls(row):
    body = ast.literal_eval(row["body_images"]) if pd.notna(row["body_images"]) else []
    og = [row["og_image"]] if pd.notna(row["og_image"]) else []
    return og + body

def get_filenames(urls):
    return [hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg" for url in urls]

df["image_urls"] = df.apply(get_urls, axis=1)
df["image_filename"] = df["image_urls"].apply(get_filenames)

In [ ]:
df

In [ ]:
import pandas as pd
import hashlib
import ast
import os

df = pd.read_csv("telex_SECONDROUND_final_articles.csv")
folder_files = set(os.listdir("telex_images_SECONDROUND"))

def get_urls(row):
    body = ast.literal_eval(row["body_images"]) if pd.notna(row["body_images"]) else []
    og = [row["og_image"]] if pd.notna(row["og_image"]) else []
    return og + body

def get_filenames(urls):
    return [hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg" for url in urls]

df["image_urls"] = df.apply(get_urls, axis=1)
df["image_filename"] = df["image_urls"].apply(get_filenames)

all_generated = set(f for row in df["image_filenames"] for f in row)

print("Generated filenames:", len(all_generated))
print("Files in folder:", len(folder_files))
print("Matched:", len(all_generated & folder_files))
print("Not matched (generated but missing in folder):", len(all_generated - folder_files))

In [ ]:
df.to_csv("telex_SECONDROUND_final_articles.csv", index=False)

In [ ]:
import pandas as pd
import ast
import os
import hashlib
from urllib.parse import urlsplit

df = pd.read_csv("magyarnemzet_SECONDROUND_final_articles.csv", low_memory=False)
folder_files = set(os.listdir("magyar_SECONDROUND_images"))

def filename_from_url(url):
    path = urlsplit(url).path
    ext = os.path.splitext(path)[1].lower()
    if ext not in [".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".tif", ".tiff"]:
        ext = ".jpg"
    h = hashlib.sha1(path.encode("utf-8")).hexdigest()[:16]
    return h + ext

def get_filenames(row):
    urls = []
    if pd.notna(row["main_image_url"]):
        urls.append(row["main_image_url"])
    if pd.notna(row["all_image_urls"]):
        try:
            imgs = ast.literal_eval(row["all_image_urls"])
            if isinstance(imgs, list):
                urls.extend(imgs)
        except:
            pass
    urls = list(dict.fromkeys(u.strip() for u in urls if isinstance(u, str) and u.strip()))
    return [filename_from_url(u) for u in urls]

df["image_filename"] = df.apply(get_filenames, axis=1)

all_generated = set(f for row in df["image_filename"] for f in row)

print("Generated filenames:", len(all_generated))
print("Files in folder:", len(folder_files))
print("Matched:", len(all_generated & folder_files))
print("Not matched:", len(all_generated - folder_files))

In [ ]:
df.to_csv("magyarnemzet_SECONDROUND_final_articles.csv", index=False)

In [ ]:
import pandas as pd
import ast
import os

def parse_filenames(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if not x:
        return []
    # try python list format
    try:
        lst = ast.literal_eval(x)
        if isinstance(lst, list):
            return [str(i).strip() for i in lst if str(i).strip()]
    except:
        pass
    # try pipe or comma separated string
    for sep in ["|", ",", ";"]:
        if sep in x:
            return [i.strip() for i in x.split(sep) if i.strip()]
    return [x]

def check_dataset(csv_path, folder_path, filename_col):
    df = pd.read_csv(csv_path, low_memory=False)
    folder_files = set(os.listdir(folder_path))

    df["_parsed_filenames"] = df[filename_col].apply(parse_filenames)
    all_in_dataset = set(f for row in df["_parsed_filenames"] for f in row)

    print(csv_path)
    print("Filenames in dataset:", len(all_in_dataset))
    print("Files in folder:", len(folder_files))
    print("Matched:", len(all_in_dataset & folder_files))
    print("Not matched (in dataset, missing in folder):", len(all_in_dataset - folder_files))
    print("Extra (in folder, not in dataset):", len(folder_files - all_in_dataset))
    print("Sample dataset filenames:", list(all_in_dataset)[:5])
    print("Sample folder filenames:", list(folder_files)[:5])
    print()

datasets_to_check = [
    ("elmundo_SECONDROUND_final_articles.csv", "elmundo_images_SECONDROUND", "image_filename"),
    ("eldiario_SECONDROUND_final_articles.csv", "eldiario_images_SECONDROUND", "image_filename"),
    ("rp_SECONDROUND_final_articles.csv", "rp_SECONDROUND_images", "image_filenames"),
    ("polityka_SECONDROUND_final_articles.csv", "polityka_SECONDROUND_images", "image_filename"),
    ("spiegel_SECONDROUND_final_articles.csv", "spiegel_images_SECONDROUND", "image_filenames"),
    ("welt_SECONDROUND_final_articles.csv", "welt_images_SECONDROUND", "image_filenames"),
]

for csv_path, folder_path, filename_col in datasets_to_check:
    check_dataset(csv_path, folder_path, filename_col)

In [ ]:
import pandas as pd
import ast
import os

def parse_filenames(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if not x:
        return []
    # try python list format
    try:
        lst = ast.literal_eval(x)
        if isinstance(lst, list):
            return [str(i).strip() for i in lst if str(i).strip()]
    except:
        pass
    # try pipe or comma separated string
    for sep in ["|", ",", ";"]:
        if sep in x:
            return [i.strip() for i in x.split(sep) if i.strip()]
    return [x]

def check_dataset(csv_path, folder_path, filename_col):
    df = pd.read_csv(csv_path, low_memory=False)
    folder_files = set(os.listdir(folder_path))

    df["_parsed_filenames"] = df[filename_col].apply(parse_filenames)
    all_in_dataset = set(f for row in df["_parsed_filenames"] for f in row)

    print(csv_path)
    print("Filenames in dataset:", len(all_in_dataset))
    print("Files in folder:", len(folder_files))
    print("Matched:", len(all_in_dataset & folder_files))
    print("Not matched (in dataset, missing in folder):", len(all_in_dataset - folder_files))
    print("Extra (in folder, not in dataset):", len(folder_files - all_in_dataset))
    print("Sample dataset filenames:", list(all_in_dataset)[:5])
    print("Sample folder filenames:", list(folder_files)[:5])
    print()
    
datasets_to_check = [
    ("magyarnemzet_SECONDROUND_final_articles.csv", "magyar_SECONDROUND_images", "image_filename"),
    ("eldiario_SECONDROUND_final_articles.csv", "eldiario_images_SECONDROUND", "image_filename"),
    ("telex_SECONDROUND_final_articles.csv", "telex_images_SECONDROUND", "image_filename"),
]

for csv_path, folder_path, filename_col in datasets_to_check:
    check_dataset(csv_path, folder_path, filename_col)

In [ ]:
## Move eldiario, Magyar and Telex
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLUMNS = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

datasets_config = [
    {
        "name": "eldiario",
        "csv": "eldiario_SECONDROUND_final_articles.csv",
        "images_folder": "eldiario_images_SECONDROUND",
        "rename": {"url": "article_url", "publish_date": "date"},
    },
    {
        "name": "magyar",
        "csv": "magyarnemzet_SECONDROUND_final_articles.csv",
        "images_folder": "magyar_SECONDROUND_images",
        "rename": {"url": "article_url", "image_urls": "image_url"},
    },
    {
        "name": "telex",
        "csv": "telex_SECONDROUND_final_articles.csv",
        "images_folder": "telex_images_SECONDROUND",
        "rename": {"url": "article_url", "publish_date": "date"},
    },
]

for config in datasets_config:
    df = pd.read_csv(config["csv"], low_memory=False)
    df = df.rename(columns=config["rename"])

    for col in TARGET_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[TARGET_COLUMNS]

    output_csv = os.path.join(OUTPUT_DIR, f"{config['name']}_full_data.csv")
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    output_images_folder = os.path.join(OUTPUT_DIR, f"{config['name']}_full_images")
    shutil.copytree(config["images_folder"], output_images_folder)

    print(config["name"], "done")

In [ ]:
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLUMNS = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

datasets_config = [

    {
        "name": "rp",
        "csv": "rp_SECONDROUND_final_articles.csv",
        "images_folder": "rp_SECONDROUND_images",
        "rename": {"url": "article_url", "image_urls": "image_url", "image_filenames": "image_filename"},
    },
    {
        "name": "polityka",
        "csv": "polityka_SECONDROUND_final_articles.csv",
        "images_folder": "polityka_SECONDROUND_images",
        "rename": {"url": "article_url", "publish_date": "date"},
    },
    {
        "name": "spiegel",
        "csv": "spiegel_SECONDROUND_final_articles.csv",
        "images_folder": "spiegel_images_SECONDROUND",
        "rename": {"section": "category", "published_time": "date", "article_text": "text",
                   "image_urls": "image_url", "image_filenames": "image_filename"},
    },
    {
        "name": "welt",
        "csv": "welt_SECONDROUND_final_articles.csv",
        "images_folder": "welt_images_SECONDROUND",
        "rename": {"url": "article_url", "published": "date", "image_urls": "image_url",
                   "image_filenames": "image_filename"},
    },
]

for config in datasets_config:
    df = pd.read_csv(config["csv"], low_memory=False)
    df = df.rename(columns=config["rename"])

    for col in TARGET_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[TARGET_COLUMNS]

    output_csv = os.path.join(OUTPUT_DIR, f"{config['name']}_full_data.csv")
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    output_images_folder = os.path.join(OUTPUT_DIR, f"{config['name']}_full_images")
    shutil.copytree(config["images_folder"], output_images_folder)

    print(config["name"], "done")

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_PATH = "spiegel_full_data.csv"
IMAGE_DIR = "."

df = pd.read_csv(DATASET_PATH)
image_dir = Path(IMAGE_DIR)

df

## Check before the analysis to identify missing data and count potential tokens

In [ ]:
# Regenerate image_filename for telex and magyar using the confirmed-working hash formulas,
# verify the match against the actual image folders, then rebuild their 3_data_analysis files.
# This avoids relying on notebook cell execution order, which earlier corrupted these two datasets.

import hashlib
import ast
import os
from urllib.parse import urlsplit
import shutil

# --- magyar: sha1 of the image URL's path, first 16 hex chars ---
df_magyar = pd.read_csv("magyarnemzet_SECONDROUND_final_articles.csv", low_memory=False)
folder_magyar = set(os.listdir("magyar_SECONDROUND_images"))

def filename_from_url(url):
    path = urlsplit(url).path
    ext = os.path.splitext(path)[1].lower()
    if ext not in [".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".tif", ".tiff"]:
        ext = ".jpg"
    h = hashlib.sha1(path.encode("utf-8")).hexdigest()[:16]
    return h + ext

def get_magyar_filenames(row):
    urls = []
    if pd.notna(row["main_image_url"]):
        urls.append(row["main_image_url"])
    if pd.notna(row["all_image_urls"]):
        try:
            imgs = ast.literal_eval(row["all_image_urls"])
            if isinstance(imgs, list):
                urls.extend(imgs)
        except Exception:
            pass
    urls = list(dict.fromkeys(u.strip() for u in urls if isinstance(u, str) and u.strip()))
    return "|".join(filename_from_url(u) for u in urls)

df_magyar["image_filename"] = df_magyar.apply(get_magyar_filenames, axis=1)

generated_magyar = set(f for names in df_magyar["image_filename"] for f in names.split("|") if f)
print("magyar — generated:", len(generated_magyar), "| matched:", len(generated_magyar & folder_magyar))

# --- telex: md5 of the image URL's path, not the full URL ---
df_telex = pd.read_csv("telex_SECONDROUND_final_articles.csv", low_memory=False)
folder_telex = set(os.listdir("telex_images_SECONDROUND"))

def telex_filename_from_url(url):
    path = urlsplit(url).path
    return hashlib.md5(path.encode("utf-8")).hexdigest() + ".jpg"

def get_telex_filenames(row):
    body = ast.literal_eval(row["body_images"]) if pd.notna(row["body_images"]) else []
    og = [row["og_image"]] if pd.notna(row["og_image"]) else []
    urls = og + body
    return "|".join(telex_filename_from_url(u) for u in urls if isinstance(u, str) and u.strip())

df_telex["image_filename"] = df_telex.apply(get_telex_filenames, axis=1)

generated_telex = set(f for names in df_telex["image_filename"] for f in names.split("|") if f)
print("telex — generated:", len(generated_telex), "| matched:", len(generated_telex & folder_telex))

In [ ]:
# Rebuild telex's image_filename using the confirmed formula (md5 of the full URL),
# then overwrite 3_data_analysis/telex_full_data.csv so it matches the actual image folder.

import hashlib

def telex_filename_from_url(url):
    return hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg"

def get_telex_filenames(row):
    body = ast.literal_eval(row["body_images"]) if pd.notna(row["body_images"]) else []
    og = [row["og_image"]] if pd.notna(row["og_image"]) else []
    urls = [u for u in (og + body) if isinstance(u, str) and u.strip()]
    return "|".join(telex_filename_from_url(u) for u in urls)

df_telex["image_filename"] = df_telex.apply(get_telex_filenames, axis=1)

# rebuild the unified 3_data_analysis file for telex with the corrected image_filename column
TARGET_COLUMNS = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

df_out = df_telex.rename(columns={"url": "article_url", "publish_date": "date"}).copy()
for col in TARGET_COLUMNS:
    if col not in df_out.columns:
        df_out[col] = pd.NA
df_out = df_out[TARGET_COLUMNS]

output_csv = os.path.join(".", "telex_full_data.csv")
df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("telex_full_data.csv rebuilt with corrected image_filename")

In [ ]:
# Rebuild magyar's image_filename the same way, using the confirmed sha1(path) formula.

df_magyar["image_filename"] = df_magyar.apply(get_magyar_filenames, axis=1)

df_out_magyar = df_magyar.rename(columns={"url": "article_url"}).copy()
for col in TARGET_COLUMNS:
    if col not in df_out_magyar.columns:
        df_out_magyar[col] = pd.NA
df_out_magyar = df_out_magyar[TARGET_COLUMNS]

output_csv = os.path.join(".", "magyar_full_data.csv")
df_out_magyar.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("magyar_full_data.csv rebuilt with corrected image_filename")

In [ ]:
# Compare text length, image availability, and image dimensions across all 7 unified datasets.
# For each dataset: reads {name}_full_data.csv, checks which image_filename entries actually
# exist in {name}_full_images/, and samples a subset of present images to estimate dimensions/size.

from pathlib import Path
import pandas as pd
from PIL import Image
import random

DATA_DIR = Path(".")
DATASET_NAMES = ["rp", "polityka", "spiegel", "welt", "telex", "magyar", "eldiario"]
SAMPLE_SIZE_FOR_DIMENSIONS = 200

summary_rows = []

for name in DATASET_NAMES:
    csv_path = DATA_DIR / f"{name}_full_data.csv"
    image_dir = DATA_DIR / f"{name}_full_images"

    df = pd.read_csv(csv_path, low_memory=False)
    df["text_len"] = df["text"].fillna("").astype(str).str.len()

    if not image_dir.exists():
        print(f"WARNING: images folder does not exist: {image_dir}")

    n_no_filenames = 0
    n_missing_on_disk = 0
    total_missing_files = 0
    n_images_per_article = []
    present_paths = []
    missing_files_this_dataset = []

    for idx, row in df.iterrows():
        filenames = row.get("image_filename")

        if pd.isna(filenames) or not str(filenames).strip():
            n_no_filenames += 1
            n_images_per_article.append(0)
            continue

        names = [n.strip() for n in str(filenames).split("|") if n.strip()]
        n_images_per_article.append(len(names))

        missing_here = []
        for n in names:
            path = image_dir / n
            if path.exists():
                present_paths.append(path)
            else:
                missing_here.append(n)
                missing_files_this_dataset.append(n)

        if missing_here:
            n_missing_on_disk += 1
            total_missing_files += len(missing_here)

    sample = random.sample(present_paths, min(SAMPLE_SIZE_FOR_DIMENSIONS, len(present_paths)))
    widths, heights, file_kb = [], [], []
    for path in sample:
        try:
            with Image.open(path) as img:
                widths.append(img.width)
                heights.append(img.height)
            file_kb.append(path.stat().st_size / 1024)
        except Exception as exc:
            print(f"  WARNING: could not read {path}: {exc}")

    summary_rows.append(
        {
            "dataset": name,
            "n_articles": len(df),
            "mean_text_len": round(df["text_len"].mean(), 1),
            "mean_n_images": round(sum(n_images_per_article) / len(n_images_per_article), 2),
            "articles_never_had_images": n_no_filenames,
            "articles_with_missing_images_on_disk": n_missing_on_disk,
            "total_missing_image_files": total_missing_files,
            "n_images_sampled": len(sample),
            "mean_image_width": round(sum(widths) / len(widths), 0) if widths else None,
            "mean_image_height": round(sum(heights) / len(heights), 0) if heights else None,
            "mean_image_kb": round(sum(file_kb) / len(file_kb), 1) if file_kb else None,
        }
    )

    # print (not save) which specific files are missing, capped to first 10 per dataset
    if missing_files_this_dataset:
        preview = missing_files_this_dataset[:10]
        more = len(missing_files_this_dataset) - len(preview)
        print(f"{name}: {total_missing_files} missing files, e.g. {preview}" + (f" (+{more} more)" if more > 0 else ""))

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 130)
print(summary_df.to_string(index=False))

In [ ]:
# Fix the text-column mapping bug for telex and magyar (article_text was never renamed to
# "text" in the original unification step, so mean_text_len came out as 0.0) while rebuilding
# their 3_data_analysis files with the corrected image_filename column.

TARGET_COLUMNS = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

# telex
df_out_telex = df_telex.rename(columns={
    "article_text": "text",
    "publication_date": "date",
}).copy()
for col in TARGET_COLUMNS:
    if col not in df_out_telex.columns:
        df_out_telex[col] = pd.NA
df_out_telex = df_out_telex[TARGET_COLUMNS]
df_out_telex.to_csv(os.path.join(".", "telex_full_data.csv"), index=False, encoding="utf-8-sig")
print("telex_full_data.csv rebuilt with corrected text + image_filename")

# magyar
df_out_magyar = df_magyar.rename(columns={
    "article_text": "text",
    "publication_date": "date",
}).copy()
for col in TARGET_COLUMNS:
    if col not in df_out_magyar.columns:
        df_out_magyar[col] = pd.NA
df_out_magyar = df_out_magyar[TARGET_COLUMNS]
df_out_magyar.to_csv(os.path.join(".", "magyar_full_data.csv"), index=False, encoding="utf-8-sig")
print("magyar_full_data.csv rebuilt with corrected text + image_filename")